# Proyecto Etapa 3. Evaluando modelos de aprendizaje en Big Data

**Instituto Tecnológico y de Estudios Superiores de Monterrey**  
**Maestría en Inteligencia Artificial Aplicada**  
**Análisis de grandes volúmenes de datos**  
**Docente:** Dr. Iván Olmos Pineda

* González Pérez, Mario Alberto — A01840153
* Palma Palacios, Christian Ricardo — A01686081
* Salas Fuentes, Abraham Israel — A01840087
* Vega Martínez, Ángel — A01377304

**Fecha:** 13 de junio de 2026

##1. Construcción de la muestra M

A continuación, se instalan las dependencias necesarias para ejecutar PySpark en el entorno de Google Colab.

In [11]:
# Instalación de Java y descarga de Spark
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [15]:
# Configuración de variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [16]:
!ls

sample_data		       spark-3.5.8-bin-hadoop3.tgz.2
spark-3.5.8-bin-hadoop3        spark-3.5.8-bin-hadoop3.tgz.3
spark-3.5.8-bin-hadoop3.tgz    spark-3.5.8-bin-hadoop3.tgz.4
spark-3.5.8-bin-hadoop3.tgz.1


In [17]:
# Inicialización de la sesión de Spark
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Habilitar formato de salida mejorado para DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

Se monta la unidad de Google Drive y se carga el archivo CSV en un DataFrame de PySpark.

In [18]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Definir ruta del archivo
file_path = "/content/drive/MyDrive/Colab Notebooks/MNA/BigData/ProyectoEtapa1/Microsoft_GUIDE_Train.csv"

Mounted at /content/drive


In [19]:
# Cargar el dataset
df = spark.read.csv(file_path, header=True, sep=",", inferSchema=True)

# Mostrar las primeras 2 filas para verificar la carga
df.limit(2).toPandas()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04 06:05:15,7,6,InitialAccess,None,TruePositive,...,None,None,5,66,None,None,None,31,6,3
1,455266534868,88,326,210035,2024-06-14 03:01:25,58,43,Exfiltration,None,FalsePositive,...,None,None,5,66,None,None,None,242,1445,10630


**Preprocesamiento**

Asegurar la calidad de los datos antes del particionamiento (muestreo) y antes del aprendizaje no supervisado.

In [20]:
from pyspark.sql.functions import col, when, count, lit

# Eliminar registros sin variable objetivo
df = df.filter(col("IncidentGrade").isNotNull())

# Eliminar duplicados
df = df.dropDuplicates()

# Eliminar timestamps inválidos
df = df.filter(col("Timestamp").isNotNull())

# Eliminar columnas >90% NULL
cols_to_drop = [
    "ActionGrouped",
    "ActionGranular",
    "EmailClusterId",
    "ResourceType",
    "Roles",
    "AntispamDirection",
    "ThreatFamily"
]
df = df.drop(*cols_to_drop)

# Imputar variables moderadamente incompletas
fill_values = {
    "MitreTechniques": "Unknown",
    "SuspicionLevel": "Unknown",
    "LastVerdict": "Unknown"
}
df = df.fillna(fill_values)

# Extraer variables temporales
from pyspark.sql.functions import hour, dayofweek, month
df = df.withColumn("EventHour", hour(col("Timestamp")))
df = df.withColumn("EventDayOfWeek", dayofweek(col("Timestamp")))
df = df.withColumn("EventMonth", month(col("Timestamp")))

In [21]:
# Reducir cardinalidad (Top-25) para aprendizaje no supervisado.
# El Top no debe ser muy pequeño porque las categorías poco frecuentes pueden contener patrones muy útiles para clustering.
top_categories = [
    row["Category"]
    for row in df.groupBy("Category")
    .count()
    .orderBy(col("count").desc())
    .limit(25)
    .collect()
]

df = df.withColumn(
    "CategoryReduced",
    when(
        col("Category").isin(top_categories),
        col("Category")
    ).otherwise("OtherCategory")
)

In [22]:
# Obtener cantidad total de registros limpios
total_records = df.count()
print(f"Cantidad total de registros limpios: {total_records:,}")

Cantidad total de registros limpios: 9,442,956


**Muestreo**

Para evitar una explosión combinatoria y estratos demasiado pequeños, se aplica una estrategia de agrupación *Top-N* sobre las variables categóricas de mayor cardinalidad.

In [23]:
# Agrupación de la variable Category (Top 10)
top_categories_sampling = [
    row["Category"]
    for row in (
        df.groupBy("Category")
          .count()
          .orderBy(col("count").desc())
          .limit(10)
          .collect()
    )
]

df = df.withColumn(
    "CategoryGroup",
    when(
        col("Category").isin(top_categories_sampling),
        col("Category")
    ).otherwise("OtherCategory")
)

# Agrupación de la variable EntityType (Top 5)
top_entity_types = [
    row["EntityType"]
    for row in (
        df.groupBy("EntityType")
          .count()
          .orderBy(col("count").desc())
          .limit(5)
          .collect()
    )
]

df = df.withColumn(
    "EntityTypeGroup",
    when(
        col("EntityType").isin(top_entity_types),
        col("EntityType")
    ).otherwise("Other")
)

Se definen las columnas que conformarán las reglas de particionamiento. En aprendizaje no supervisado no deberíamos utilizar la variable objetivo "IncidentGrade" para construir la muestra.

In [24]:
# Definir columnas de particionamiento
partition_columns = [
    "CategoryGroup",
    "EntityTypeGroup",
    "EvidenceRole"
]

Muestreo estratificado desproporcional sin reemplazo, utilizando como estratos las combinaciones observadas empíricamente de las variables: CategoryGroup, EntityTypeGroup y EvidenceRole.

Como el objetivo es mejorar el clustering (aprendizaje automático no supervisado), conviene el muestreo desproporcional porque en ciberseguridad existen patrones raros que pueden desaparecer cuando se toma solamente un porcentaje fijo (proporcional). K-Means y GMM suelen encontrar mejores grupos cuando esos patrones minoritarios sobreviven al muestreo.

In [25]:
from pyspark.sql.functions import concat_ws

# =====================================================
# 1. Crear columna estrato (partición)
# =====================================================

df = df.withColumn(
    "Stratum",
    concat_ws(
        "_",
        col("CategoryGroup"),
        col("EntityTypeGroup"),
        col("EvidenceRole")
    )
)

# =====================================================
# 2. Estadísticas de Estratos
# =====================================================

strata_counts = (
    df.groupBy("Stratum")
      .count()
      .cache()
)

print("Número de estratos observados:")
print(strata_counts.count())


# =====================================================
# 3. Muestreo Adaptativo
# =====================================================

# Protección de estratos pequeños
fractions = {}

for row in strata_counts.collect():

    n = row["count"]

    if n < 100:
        fractions[row["Stratum"]] = 1.0

    elif n < 1000:
        fractions[row["Stratum"]] = 0.30

    else:
        fractions[row["Stratum"]] = 0.10

# =====================================================
# 4. Extraer muestra estratificada M
# =====================================================

seed_value = 42

sample_M = df.sampleBy(
    "Stratum",
    fractions=fractions,
    seed=seed_value
)

# =====================================================
# 5. Persistir M en memoria
# =====================================================

from pyspark import StorageLevel

sample_M.persist(
    StorageLevel.MEMORY_AND_DISK
)

_ = sample_M.count()

# =====================================================
# 6. Verificar tamaño de muestra
# =====================================================

sample_count = sample_M.count()

print(f"Total registros dataset original: {total_records:,}")
print(f"Total registros muestra M: {sample_count:,}")
print(f"Porcentaje real: {(sample_count/total_records)*100:.4f}%")

Número de estratos observados:
97
Total registros dataset original: 9,442,956
Total registros muestra M: 945,732
Porcentaje real: 10.0152%


Verificación de representatividad. La muestra mantiene aproximadamente las proporciones originales.

In [26]:
# Distribución original
original_distribution = df.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "OriginalCount")

# Distribución de la muestra
sample_distribution = sample_M.groupBy(*partition_columns) \
    .count() \
    .withColumnRenamed("count", "SampleCount")

# Comparación
comparison = original_distribution.join(
    sample_distribution,
    on=partition_columns,
    how="left"
)

comparison = comparison.fillna(0)
comparison = comparison.withColumn(
    "SamplePct",
    (col("SampleCount") / col("OriginalCount")) * 100
)
comparison.orderBy(col("OriginalCount").desc()).show(3, truncate=False)

+-------------+---------------+------------+-------------+-----------+------------------+
|CategoryGroup|EntityTypeGroup|EvidenceRole|OriginalCount|SampleCount|SamplePct         |
+-------------+---------------+------------+-------------+-----------+------------------+
|InitialAccess|Other          |Related     |1268624      |126973     |10.00871810717754 |
|InitialAccess|User           |Impacted    |935112       |93662      |10.016126410526226|
|InitialAccess|Ip             |Related     |853324       |85167      |9.980616975498169 |
+-------------+---------------+------------+-------------+-----------+------------------+
only showing top 3 rows



La muestra M original presentaba una estrategia de muestreo estratificado proporcional sin reemplazo, que era adecuada; sin embargo, se identificaron oportunidades de mejora. La principal consiste en eliminar la variable IncidentGrade de la definición de los estratos, dado que corresponde a la variable objetivo y su utilización introduce sesgo en un escenario de aprendizaje no supervisado. Asimismo, se propone incrementar la granularidad de las variables categóricas agrupadas y preservar estratos pequeños mediante fracciones adaptativas. Estas mejoras permitirían capturar una mayor diversidad de patrones presentes en el conjunto de datos y potencialmente mejorar la calidad de los agrupamientos obtenidos mediante K-Means y GMM, reflejándose en métricas como el Silhouette Score.

##2. Construcción Train-Test

Ahora debemos garantizar que Train y Test sean representativos de la muestra M, manteniendo exactamente la misma filosofía de muestreo utilizada en la construcción de M.
- La consigna indica que M={Mi} donde cada Mi representa un estrato definido por las variables de caracterización.
- En este caso, los estratos fueron definidos mediante: CategoryGroup, EntityTypeGroup, EvidenceRole y materializados en: Stratum.
- El objetivo es lograr que: Train ∩ Test = ∅ y Train ∪ Test = M.

La literatura de Machine Learning para datasets grandes recomienda:

| Tamaño dataset   | División recomendada |
| ---------------- | -------------------- |
| < 10,000         | 80/20                |
| 10,000 – 100,000 | 75/25                |
| > 100,000        | 70/30                |

La muestra M contiene 945,732 registros, por lo tanto el porcentaje de división a utilizar será Train = 70% y Test = 30%.

- Esto significa tener suficiente información para aprender patrones, contar con un conjunto Test muy grande que permitirá una menor varianza en métricas de evaluación.

No utilizaremo sample_M.randomSplit([0.7,0.3]) porque puede alterar la distribución de los estratos y no garantiza representatividad dentro de cada Mi.

Usaremos nuevamente muestreo estratificado, exactamente la misma lógica empleada para construir M. Cada estrato debe dividirse en 70% Train y 30% Test.

In [27]:
# =====================================================
# Verificar número de estratos presentes en M
# =====================================================

num_strata = (
    sample_M
    .select("Stratum")
    .distinct()
    .count()
)

print(f"Estratos observados en M: {num_strata}")

Estratos observados en M: 97


In [28]:
# =====================================================
# Obtener lista de estratos
# =====================================================

strata_list = [
    row["Stratum"]
    for row in (
        sample_M
        .select("Stratum")
        .distinct()
        .collect()
    )
]

print(f"Total estratos: {len(strata_list)}")

Total estratos: 97


In [29]:
# =====================================================
# Fracción para Train
# =====================================================

train_fraction = 0.70

fractions_train = {
    stratum: train_fraction
    for stratum in strata_list
}

In [30]:
# Crear identificador único antes del muestreo
from pyspark.sql.functions import monotonically_increasing_id

sample_M = sample_M.withColumn(
    "row_id",
    monotonically_increasing_id()
)

In [31]:
# =====================================================
# Construcción de Train
# =====================================================

seed_value = 42

train_df = sample_M.sampleBy(
    col="Stratum",
    fractions=fractions_train,
    seed=seed_value
)

Construir Test como complemento. No usaremos sampleBy() para garantizar Train ∩ Test = ∅

In [32]:
# Construir Test
test_df = (
    sample_M
    .join(
        train_df.select("row_id"),
        on="row_id",
        how="left_anti"
    )
)

In [33]:
# Persistencia
from pyspark import StorageLevel

train_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

test_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

_ = train_df.count()
_ = test_df.count()

Validar Conjuntos

In [34]:
# Validar tamaño de los conjuntos
train_count = train_df.count()
test_count = test_df.count()
M_count = sample_M.count()

print(f"M: {M_count:,}")
print(f"Train: {train_count:,}")
print(f"Test: {test_count:,}")

print(
    f"Train %: {(train_count/M_count)*100:.2f}"
)

print(
    f"Test %: {(test_count/M_count)*100:.2f}"
)

# Validar exhaustividad
print("Train ∪ Test = M : ", train_count + test_count == M_count)

# Validar exclusión mutua
intersection = (
    train_df.select("row_id")
    .intersect(
        test_df.select("row_id")
    )
    .count()
)
print("Intersección de conjuntos:",intersection)

M: 945,732
Train: 662,106
Test: 283,626
Train %: 70.01
Test %: 29.99
Train ∪ Test = M :  True
Intersección de conjuntos: 0


Validación de representatividad de los conjuntos en cada estrato

In [36]:
from pyspark.sql.functions import col, round

# =====================================================
# Distribución de estratos en M
# =====================================================

m_dist = (
    sample_M
    .groupBy(
        "CategoryGroup",
        "EntityTypeGroup",
        "EvidenceRole"
    )
    .count()
    .withColumnRenamed("count", "M_Count")
)

# =====================================================
# Distribución de estratos en Train
# =====================================================

train_dist = (
    train_df
    .groupBy(
        "CategoryGroup",
        "EntityTypeGroup",
        "EvidenceRole"
    )
    .count()
    .withColumnRenamed("count", "Train_Count")
)

# =====================================================
# Distribución de estratos en Test
# =====================================================

test_dist = (
    test_df
    .groupBy(
        "CategoryGroup",
        "EntityTypeGroup",
        "EvidenceRole"
    )
    .count()
    .withColumnRenamed("count", "Test_Count")
)

# =====================================================
# Consolidar resultados
# =====================================================

validation_df = (
    m_dist
    .join(
        train_dist,
        ["CategoryGroup", "EntityTypeGroup", "EvidenceRole"],
        "left"
    )
    .join(
        test_dist,
        ["CategoryGroup", "EntityTypeGroup", "EvidenceRole"],
        "left"
    )
)

# =====================================================
# Calcular porcentajes
# =====================================================

validation_df = (
    validation_df
    .withColumn(
        "Train_Pct",
        round(
            (col("Train_Count") / col("M_Count")) * 100,
            2
        )
    )
    .withColumn(
        "Test_Pct",
        round(
            (col("Test_Count") / col("M_Count")) * 100,
            2
        )
    )
    .withColumn(
        "Total_Pct",
        round(
            (
                (col("Train_Count") + col("Test_Count"))
                / col("M_Count")
            ) * 100,
            2
        )
    )
)

# =====================================================
# Mostrar primeros 10 estratos
# =====================================================

validation_df \
    .orderBy(col("M_Count").desc()) \
    .show(10, truncate=False)

+-----------------+---------------+------------+-------+-----------+----------+---------+--------+---------+
|CategoryGroup    |EntityTypeGroup|EvidenceRole|M_Count|Train_Count|Test_Count|Train_Pct|Test_Pct|Total_Pct|
+-----------------+---------------+------------+-------+-----------+----------+---------+--------+---------+
|InitialAccess    |Other          |Related     |126973 |89060      |37913     |70.14    |29.86   |100.0    |
|InitialAccess    |User           |Impacted    |93662  |65156      |28506     |69.57    |30.43   |100.0    |
|InitialAccess    |Ip             |Related     |85167  |59789      |25378     |70.2     |29.8    |100.0    |
|Impact           |Ip             |Related     |68378  |47797      |20581     |69.9     |30.1    |100.0    |
|Exfiltration     |MailMessage    |Impacted    |67475  |47306      |20169     |70.11    |29.89   |100.0    |
|InitialAccess    |MailMessage    |Related     |49181  |34224      |14957     |69.59    |30.41   |100.0    |
|InitialAccess    |

##3. Selección de métricas para medir calidad de resultados

Debemos decidir cómo evaluar objetivamente la calidad de los clusters obtenidos sobre un dataset SOC de 945 mil registros.

##Métricas Internas

**Silhouette**

Anteriorente se utilizó Silhouette Score porque:
- Mide cohesión y separación entre clusters.
- No requiere etiquetas reales.
- Está implementada directamente en PySpark.
- Es ampliamente utilizada en clustering de anomalías y telemetría.
- Funciona correctamente tanto para K-Means como para GMM.

Obteniéndose un Silhouette Score en TEST de 0.3317. Lo que significa que existe estructura de agrupamiento, pero los clusters se solapan.

Silhouette favorece clusters convexos, clusters aproximadamente esféricos y tamaños similares. Por ello, K-Means suele verse favorecido y GMM ligeramente favorecido, mientras que otros algoritmos podrían verse penalizados.

Además, el dataset tiene esta naturaleza:
- Múltiples organizaciones.
- Múltiples tipos de ataque.
- Múltiples entidades.
- Fases MITRE ATT&CK.
- Comportamiento SOC real.
- Gran presencia de ruido y falsos positivos.

Esto sugiere que los clusters reales probablemente no sean esféricos, tengan diferentes densidades y tengan diferentes tamaños. Por ello Silhouette sola no basta.

Recordemos que el valor del Silhouette Score se interpreta así:

- Cercano a 1 → clusters bien definidos.
- Cercano a 0 → clusters superpuestos.
- Negativo → clustering deficiente.


**Davies-Bouldin Index (DBI)**

- Mide: compactación intra-cluster vs separación inter-cluster.
- Ventaja: Funciona muy bien cuando los clusters tienen distintos tamaños o los clusters tienen distintas densidades. Precisamente lo que esperamos en este dataset.
- Interpretación:

| DBI       | Calidad   |
| --------- | --------- |
| < 0.5     | Excelente |
| 0.5 - 1.0 | Buena     |
| 1.0 - 2.0 | Aceptable |
| > 2.0     | Mala      |


**Índice de Calinski-Harabasz (CH)**

- Mide: Separación entre clusters / Dispersión interna
- Ventaja: Es muy eficiente computacionalmente lo que importa para el tamaño de este dataset.
- Interpretación: Mayor es mejor y no tiene límite superior. Se utiliza mucho para elegir el k óptimo


**Índice de Dunn**

- Teóricamente excelente.
- Mide: Distancia mínima entre clusters / Diámetro máximo de clusters.
- Problema: Costo computacional enorme. Para este dataset puede ser prácticamente inviable.
- Interpretación: Mayor es mejor.

**Within Set Sum of Squared Errors (WSSSE)**
- Fundamental para K-Means.
- Spark la devuelve automáticamente.
- Mide: distancia de los puntos a su centroide.
- Interpretación: Menor es mejor.
- Sirve para el Método del Codo.

**BIC y AIC (solo para GMM)**
- Si se utiliza el modelo GMM se debería considerar: BIC y AIC.
- Ventaja: Permiten seleccionar el número óptimo de componentes gaussianas.
- En GMM son más importantes que Silhouette.

##Métrica externa

Aunque estamos haciendo clustering, existe la variable objetivo "IncidentGrade" que podríamos usarla para evaluar (no para entrenar). Las siguientes métricas son valiosas en este proyecto, porque nos permiten responder: ¿Los clusters descubiertos tienen relación con la clasificación SOC?

**Homogeneity Score**

- Mide: ¿Los clusters contienen principalmente una clase?

**Normalized Mutual Information (NMI)**

- Mide: Cuánta información comparten Cluster e IncidentGrade.

**Adjusted Rand Index (ARI)**

- Compara: Cluster vs IncidentGrade


##4. Entrenamiento de Modelos de Aprendizaje

Finalmente se eligieron estos dos modelos de aprendizaje no supervisado:

**K-Means**

- Escala muy bien a cientos de miles o millones de registros.
- Spark está altamente optimizado para K-Means distribuido.
- Tiene convergencia relativamente rápida.
- Permite explorar múltiples valores de K.
- Funciona adecuadamente cuando se usan variables codificadas mediante Frequency Encoding.

**Bisecting K-Means**

- Porque el dataset GUIDE tiene una estructura potencialmente jerárquica.
- Disponible en PySpark MLlib.
- Muy escalable.
- Más estable que K-Means tradicional.
- Menor sensibilidad a centroides iniciales.
- Frecuentemente obtiene mejores valores de Silhouette en datasets grandes.

GMM se descarta porque para este proyecto presenta varios problemas:
- Escalabilidad. Su complejidad crece mucho más rápido que K-Means.
- Memoria. GMM consume bastante más memoria y en Google Colab suele convertirse en el cuello de botella.
- Variables Frequency Encoded. Estas variables no generan distribuciones gaussianas naturales. Por tanto, una de las principales hipótesis de GMM queda debilitada.
- Relación costo-beneficio. La mejora de GMM suele ser marginal respecto al incremento de tiempo de entrenamiento.


###4.1. Estrategia general de entrenamiento

- M: Train (70%) y Test (30%)
- Train:
  - Frequency Encoding
  - VectorAssembler
  - StandardScaler y PCA
  - K-Means
  - Bisecting K-Means
- Evaluación:
  - Silhouette
  - Davies-Bouldin
  - Calinski-Harabasz
  - Comparación Train-Test

###4.2. Variables predictoras

Se utilizarán variables relevantes para clustering de comportamiento en ciberseguridad.

In [37]:
# ==========================================
# Variables predictoras
# ==========================================

categorical_cols = [
    "CategoryReduced",
    "MitreTechniques",
    "EntityType",
    "EvidenceRole",
    "SuspicionLevel",
    "LastVerdict"
]

numeric_cols = [
    "EventHour",
    "EventDayOfWeek",
    "EventMonth"
]

###4.3 Transformación de variables (Frequency Encoding)

In [38]:
# =====================================================
# FREQUENCY ENCODING (solo usando Train)
# =====================================================

for c in categorical_cols:

    freq_df = (
        train_df
        .groupBy(c)
        .count()
        .withColumnRenamed(
            "count",
            f"{c}_freq"
        )
    )

    train_df = (
        train_df
        .join(freq_df, c, "left")
    )

    test_df = (
        test_df
        .join(freq_df, c, "left")
    )

    train_df = train_df.fillna(
        {f"{c}_freq": 0}
    )

    test_df = test_df.fillna(
        {f"{c}_freq": 0}
    )

###4.4. Construcción del vector de características

In [39]:
from pyspark.ml.feature import VectorAssembler

# =====================================================
# FEATURES
# =====================================================

feature_cols = (
    [f"{c}_freq" for c in categorical_cols]
    + numeric_cols
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

In [40]:
# =====================================================
# Verificación
# =====================================================

train_df.select("features").show(5, truncate=False)

+------------------------------------------------------------------+
|features                                                          |
+------------------------------------------------------------------+
|[57960.0,379771.0,131335.0,297570.0,100748.0,98161.0,12.0,6.0,6.0]|
|[57960.0,379771.0,47874.0,364536.0,100748.0,98161.0,2.0,5.0,6.0]  |
|[57960.0,379771.0,47874.0,364536.0,100748.0,98161.0,18.0,6.0,6.0] |
|[57960.0,379771.0,49009.0,297570.0,100748.0,98161.0,16.0,2.0,6.0] |
|[57960.0,2323.0,152426.0,364536.0,561325.0,505901.0,3.0,6.0,6.0]  |
+------------------------------------------------------------------+
only showing top 5 rows



###4.5. Escalamiento

In [41]:
from pyspark.ml.feature import StandardScaler

# =====================================================
# STANDARD SCALER
# =====================================================

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(train_df)

train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

# =====================================================
# Verificación
# =====================================================

train_df.select("features", "scaledFeatures").show(5, truncate=False)

+------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                          |scaledFeatures                                                                                                                                                                  |
+------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[57960.0,379771.0,131335.0,297570.0,100748.0,98161.0,12.0,6.0,6.0]|[-0.9519770943649463,0.8532240028848797,0.8547076679548921,-1.1068157013883138,-2.359808960636911,-1.65364453069431,-0.024126474617715225,1.2037456903530366,0.3115545014936665]|
|[57960.0,379771

In [65]:
from pyspark.ml.feature import PCA

# =====================================================
# PCA
# =====================================================

pca = PCA(
    k=5,
    inputCol="scaledFeatures",
    outputCol="pcaFeatures"
)

pca_model = pca.fit(train_df)

train_df = pca_model.transform(train_df)
test_df = pca_model.transform(test_df)

###4.6 Entrenamiento y validación de modelos

In [66]:
from pyspark.ml.evaluation import ClusteringEvaluator

# =====================================================
# EVALUADOR
# =====================================================

evaluator = ClusteringEvaluator(
    featuresCol="pcaFeatures",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

Entrenamiento y Validación K-Means

In [70]:
from pyspark.ml.clustering import KMeans

# ==========================================
# K-MEANS
# ==========================================

k_values = [5,8]

best_kmeans_model = None
best_kmeans_score = -1
best_k = None

for k in k_values:

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="pcaFeatures",
        predictionCol="prediction",
        maxIter=20  # mejora performance
    )

    model = kmeans.fit(train_df)

    # TRAIN
    train_pred = model.transform(train_df)

    train_silhouette = evaluator.evaluate(train_pred)

    # TEST
    test_pred = model.transform(test_df)

    test_silhouette = evaluator.evaluate(test_pred)

    diff_pct = abs(train_silhouette - test_silhouette)

    print("=" * 50)
    print(f"KMEANS - K={k}")
    print(f"Train Silhouette: {train_silhouette:.4f}")
    print(f"Test Silhouette: {test_silhouette:.4f}")
    print(f"Diferencia absoluta: {diff_pct:.4f}")

    # Verificación sobreentrenamiento
    if diff_pct < 0.03:
        print("El modelo NO está sobreentrenado.")
    else:
        print("Posible sobreentrenamiento.")

    # Verificación subentrenamiento (> 0.50 en telemetría SOC indica clusters relativamente bien definidos)
    if train_silhouette < 0.50 and test_silhouette < 0.50:
        print("Posible subentrenamiento.")
    else:
        print("No hay subentrenamiento.")

    # Mejor modelo
    if test_silhouette > best_kmeans_score:
        best_kmeans_score = test_silhouette
        best_kmeans_model = model
        best_k = k

KMEANS - K=5
Train Silhouette: 0.4277
Test Silhouette: 0.4267
Diferencia absoluta: 0.0010
El modelo NO está sobreentrenado.
Posible subentrenamiento.
KMEANS - K=8
Train Silhouette: 0.4054
Test Silhouette: 0.4037
Diferencia absoluta: 0.0017
El modelo NO está sobreentrenado.
Posible subentrenamiento.


Entrenamiento y Validación Bisecting K-Means

In [69]:
from pyspark.ml.clustering import BisectingKMeans

# ==========================================
# BISECTING KMEANS
# ==========================================

k_values = [8,9]

best_bkm_model = None
best_bkm_score = -1
best_bkm_k = None

for k in k_values:

    bkm = BisectingKMeans(
        k=k,
        seed=42,
        maxIter=20,
        #minDivisibleClusterSize=100,
        featuresCol="pcaFeatures",
        predictionCol="prediction"
    )

    model = bkm.fit(train_df)

    # TRAIN
    train_pred = model.transform(train_df)

    print(f"Clusters generados con BKM K={k}:", len(model.clusterCenters()))

    train_silhouette = evaluator.evaluate(train_pred)

    # TEST
    test_pred = model.transform(test_df)

    test_silhouette = evaluator.evaluate(test_pred)

    diff_pct = abs(train_silhouette - test_silhouette)

    print("=" * 50)
    print(f"BKM - K={k}")
    print(f"Train Silhouette: {train_silhouette:.4f}")
    print(f"Test Silhouette: {test_silhouette:.4f}")
    print(f"Diferencia absoluta: {diff_pct:.4f}")

    # Verificación sobreentrenamiento
    if diff_pct < 0.03:
        print("El modelo NO está sobreentrenado.")
    else:
        print("Posible sobreentrenamiento.")

    # Verificación subentrenamiento (> 0.50 en telemetría SOC indica clusters relativamente bien definidos)
    if train_silhouette < 0.50 and test_silhouette < 0.50:
      print("Posible subentrenamiento.")
    else:
      print("No hay subentrenamiento.")

    # Mejor modelo
    if test_silhouette > best_bkm_score:
        best_bkm_score = test_silhouette
        best_bkm_model = model
        best_bkm_k = k

Clusters generados con BKM K=8: 8
BKM - K=8
Train Silhouette: 0.3171
Test Silhouette: 0.3176
Diferencia absoluta: 0.0006
El modelo NO está sobreentrenado.
Posible subentrenamiento.
Clusters generados con BKM K=9: 9
BKM - K=9
Train Silhouette: 0.3282
Test Silhouette: 0.3281
Diferencia absoluta: 0.0001
El modelo NO está sobreentrenado.
Posible subentrenamiento.


###4.7 Selección del mejor modelo

In [71]:
print("=" * 60)
print("MEJOR MODELO KMEANS")
print(f"K óptimo: {best_k}")
print(f"Validation Silhouette: {best_kmeans_score:.4f}")

print("=" * 60)
print("MEJOR MODELO BISECTING")
print(f"K óptimo: {best_bkm_k}")
print(f"Validation Silhouette: {best_bkm_score:.4f}")

# Selección
if best_kmeans_score > best_bkm_score:

    final_model = best_kmeans_model
    final_model_name = "K-Means"

else:

    final_model = best_bkm_model
    final_model_name = "Bisecting K-Means"

print("=" * 60)
print(f"MEJOR MODELO FINAL: {final_model_name}")

MEJOR MODELO KMEANS
K óptimo: 5
Validation Silhouette: 0.4267
MEJOR MODELO BISECTING
K óptimo: 9
Validation Silhouette: 0.3281
MEJOR MODELO FINAL: K-Means


###4.8 Evaluación final

Luego del proceso de entrenamiento y validación de los modelos de aprendizaje no supervisado, se determinó que el mejor algoritmo fue K-Means con K=5, obteniendo un Test Silhouette Score = 0.4267

In [72]:
# ==========================================
# Evaluación final
# ==========================================

# Predicciones sobre TEST
test_pred = final_model.transform(test_df)

# ==========================================
# SILHOUETTE
# ==========================================

test_silhouette = evaluator.evaluate(test_pred)

print("=" * 60)
print(f"Modelo final: {final_model_name}")
print(f"Silhouette Score TEST: {test_silhouette:.4f}")

Modelo final: K-Means
Silhouette Score TEST: 0.4267


Preparar datos para DBI y CH

In [73]:
# ==========================================
# EXTRAER PCA + CLUSTER
# ==========================================

from pyspark.ml.functions import vector_to_array

metrics_df = (
    test_pred
    .select(
        vector_to_array("pcaFeatures").alias("features"),
        "prediction"
    )
)

# ==========================================
# SPARK -> PANDAS
# ==========================================

pdf = metrics_df.toPandas()


# ==========================================
# MATRIZ DE FEATURES
# ==========================================

import numpy as np

X = np.vstack(pdf["features"])

labels = pdf["prediction"].values

In [74]:
# ==========================================
# DBI y CH
# ==========================================

from sklearn.metrics import (
    davies_bouldin_score,
    calinski_harabasz_score
)

dbi = davies_bouldin_score(
    X,
    labels
)

ch = calinski_harabasz_score(
    X,
    labels
)

print(f"Davies-Bouldin Index: {dbi:.6f}")
print(f"Calinski-Harabasz Index: {ch:.6f}")

Davies-Bouldin Index: 1.352579
Calinski-Harabasz Index: 100224.363789


Interpretación del modelo final (K-Means)
- K-Means intenta responder la siguiente pregunta: ¿Puedo dividir mis datos en K grupos de tal forma que los registros dentro de cada grupo sean lo más parecidos posible y los grupos entre sí sean lo más diferentes posible?
- Elegir un K=5 significa que la mejor estructura que se encontró en estos eventos de seguridad consiste en 5 grupos (clusters) principales. Son 5 patrones de comportamiento diferenciados.
- El significado de un cluster debe interpretarse posteriormente.


In [77]:
# Cantidad de registros por cluster
test_pred.groupBy("prediction") \
           .count() \
           .orderBy("prediction") \
           .show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0|65421|
|         1|20737|
|         2|71562|
|         3|42780|
|         4|83126|
+----------+-----+



In [81]:
# Qué caracteriza a cada cluster

from pyspark.sql import functions as F

# Distribución de CategoryReduced por cluster
test_pred.groupBy(
    "prediction",
    "CategoryReduced"
).count().orderBy(
    "prediction",
    F.desc("count")
).show(40, truncate=False)

+----------+-------------------+-----+
|prediction|CategoryReduced    |count|
+----------+-------------------+-----+
|0         |InitialAccess      |51140|
|0         |CredentialAccess   |3291 |
|0         |SuspiciousActivity |2305 |
|0         |CommandAndControl  |2278 |
|0         |Impact             |2243 |
|0         |Exfiltration       |1222 |
|0         |Execution          |952  |
|0         |Discovery          |823  |
|0         |Persistence        |729  |
|0         |LateralMovement    |233  |
|0         |DefenseEvasion     |91   |
|0         |Collection         |81   |
|0         |Malware            |19   |
|0         |PrivilegeEscalation|10   |
|0         |Exploit            |2    |
|0         |UnwantedSoftware   |1    |
|0         |Ransomware         |1    |
|1         |Impact             |17321|
|1         |CommandAndControl  |1995 |
|1         |CredentialAccess   |283  |
|1         |Discovery          |275  |
|1         |InitialAccess      |239  |
|1         |Exfiltration 

In [82]:
# Distribución de IncidentGrade por cluster
test_pred.groupBy(
    "prediction",
    "IncidentGrade"
).count().orderBy(
    "prediction",
    F.desc("count")
).show(100, truncate=False)

+----------+--------------+-----+
|prediction|IncidentGrade |count|
+----------+--------------+-----+
|0         |TruePositive  |34386|
|0         |BenignPositive|16195|
|0         |FalsePositive |14840|
|1         |BenignPositive|8800 |
|1         |TruePositive  |7637 |
|1         |FalsePositive |4300 |
|2         |TruePositive  |31278|
|2         |BenignPositive|23596|
|2         |FalsePositive |16688|
|3         |BenignPositive|26864|
|3         |TruePositive  |9376 |
|3         |FalsePositive |6540 |
|4         |BenignPositive|48104|
|4         |FalsePositive |18480|
|4         |TruePositive  |16542|
+----------+--------------+-----+



##5. Análisis de Resultados

**5.1 Evaluación de estabilidad de los modelos**

El primer aspecto a analizar es la capacidad de generalización de los modelos entrenados. Para ello se compararon los resultados obtenidos en los conjuntos Train y Test mediante el Silhouette Score.

| Modelo      |  Train |   Test |    GAP |
| ----------- | -----: | -----: | -----: |
| K-Means K=5 | 0.4277 | 0.4267 | 0.0010 |
| K-Means K=8 | 0.4054 | 0.4037 | 0.0017 |

| Modelo  |  Train |   Test |    GAP |
| ------- | -----: | -----: | -----: |
| BKM K=8 | 0.3171 | 0.3176 | 0.0006 |
| BKM K=9 | 0.3282 | 0.3281 | 0.0001 |

Los cuatro modelos presentan diferencias extremadamente pequeñas entre entrenamiento y prueba, todas inferiores al 0.2%.

Por lo tanto, puede afirmarse que ninguno de los modelos presenta evidencia de sobreajuste. Esto indica que los patrones descubiertos durante el entrenamiento se mantienen prácticamente intactos al ser aplicados sobre datos no utilizados para construir el modelo.

La estabilidad observada resulta particularmente relevante considerando que el conjunto de datos contiene aproximadamente 945 mil registros distribuidos en múltiples categorías de incidentes de seguridad, organizaciones y tipos de evidencia.

**5.2 Comparación entre K-Means y Bisecting K-Means**

El principal criterio de comparación fue el Silhouette Score.

| Modelo      | Silhouette Test |
| ----------- | --------------: |
| K-Means K=5 |          0.4267 |
| K-Means K=8 |          0.4037 |
| BKM K=9     |          0.3281 |
| BKM K=8     |          0.3176 |

Los resultados muestran que K-Means supera consistentemente a Bisecting K-Means.

Una diferencia cercana a 0.10 en Silhouette Score es significativa en problemas de clustering y sugiere que K-Means genera agrupamientos más compactos y mejor separados para este conjunto de datos.

Este resultado puede explicarse por la naturaleza de las variables utilizadas:

- Frequency Encoding sobre variables categóricas.
- Variables temporales discretas.
- Datos posteriormente estandarizados y reducidos mediante PCA.

**5.3 Evaluación del modelo final seleccionado**

| Métrica           |     Valor |
| ----------------- | --------: |
| Silhouette        |    0.4267 |
| Davies-Bouldin    |    1.3526 |
| Calinski-Harabasz | 100224.36 |

**Interpretación del Silhouette Score**

El valor alcanzado se ubica dentro del rango considerado aceptable y cercano al nivel bueno. Esto indica que:
- Los registros dentro de cada cluster presentan similitud razonable.
- Existe separación entre grupos.
- Todavía hay cierto solapamiento entre clusters.

Dado que el dataset representa eventos reales de ciberseguridad, es esperable encontrar zonas grises entre categorías de incidentes, falsos positivos y eventos benignos.

**Interpretación del Davies-Bouldin Index**

Recordando que valores menores son mejores y cero representa separación perfecta. El resultado obtenido indica una calidad moderada de agrupamiento.

Los clusters presentan separación aceptable, aunque todavía existe cierta proximidad entre algunos grupos. Esto es coherente con el comportamiento observado en el Silhouette Score.

**Interpretación del Calinski-Harabasz Index**

Esta métrica no posee un límite superior definido y debe interpretarse de forma comparativa. Su valor elevado indica existe alta dispersión entre clusters,
buena compactación interna y estructura global consistente.

El hecho de obtener un valor tan alto sobre un conjunto cercano al millón de registros constituye una evidencia adicional de que la segmentación descubierta por K-Means es significativa.

**5.4 Fortalezas del modelo obtenido**
- Excelente capacidad de generalización
- Escalabilidad. El modelo fue capaz de procesar aproximadamente 945,732 registros sin requerir técnicas complejas de reducción extrema de datos.
- Interpretabilidad. Los centroides generados por K-Means permiten analizar posteriormente tipos de ataque predominantes, técnicas MITRE asociadas, roles de evidencia y niveles de sospecha. Lo anterior facilita la interpretación por parte de analistas SOC.
- Consistencia entre métricas. Las tres métricas utilizadas (Silhouette, DBI y CH) apuntan a la misma conclusión, existe una estructura de agrupamiento real y consistente dentro del dataset.

**5.5 Áreas de oportunidad**
- Calidad moderada de los clusters. Un Silhouette de 0.4267 indica que aún existe superposición entre grupos. Esto sugiere que algunos incidentes comparten características similares y no pueden separarse completamente mediante las variables utilizadas.
- Limitaciones del Frequency Encoding. Reduce dimensionalidad, pero también puede perder información semántica importante. Ejemplo, dos técnicas MITRE diferentes pueden recibir frecuencias similares.
- Variables temporales limitadas. Podrían incorporarse variables temporales adicionales que capturen patrones de comportamiento más complejos.
- Posible mejora mediante nuevas variables. El dataset contiene atributos potencialmente relevantes que no fueron incorporados al modelo final, tales como: Información organizacional (OrgId agrupado), Características derivadas de técnicas MITRE, Variables agregadas por entidad o incidente.

**5.6 Reflexión final**

Los resultados obtenidos demuestran que el dataset Microsoft GUIDE contiene patrones de comportamiento suficientemente consistentes para ser identificados mediante técnicas de aprendizaje no supervisado. Entre los modelos evaluados, K-Means con cinco clusters presentó el mejor equilibrio entre calidad de agrupamiento, estabilidad y eficiencia computacional. El modelo alcanzó un Silhouette Score de 0.4267, un Davies-Bouldin Index de 1.3526 y un Calinski-Harabasz Index de 100224.36, evidenciando agrupamientos razonablemente compactos y separados. Asimismo, la diferencia prácticamente nula entre los resultados de entrenamiento y prueba confirma la robustez del modelo y su capacidad para generalizar sobre nuevos datos. Aunque existen oportunidades de mejora relacionadas con la ingeniería de características y la incorporación de nuevas variables, los resultados alcanzados permiten concluir que el modelo es capaz de descubrir estructuras relevantes dentro de los eventos de seguridad analizados y constituye una base sólida para futuras tareas de segmentación y análisis de incidentes en entornos SOC.

**Repositorio del proyecto:** [GitHub - Etapa 3](https://github.com/cpalma-dev/BigData/blob/main/Etapa_3_Equipo37.ipynb)
